google collab depedencies

In [1]:
!pip -q install bertopic
!pip -q install sastrawi
!pip -q install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 25.6 MB/s eta 0:00:00


In [2]:
!git clone -q -b gavriel-thesis https://github.com/ranslemus/topic_modeling_KBMI4.git
%cd topic_modeling_KBMI4

/content/topic_modeling_KBMI4


In [3]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.express as px

from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from hdbscan.validity import validity_index

# for linux
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN

# for windows
# import umap as UMAP
# import hdbscan as HDBSCAN

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Device : cuda
GPU : Tesla T4


In [46]:
df = pd.read_csv("data/preprocessed_data_downsampled.csv")
df = df[df['year'] == 2025]
df.head()

,reviewId,bank,score,year,text
3,629f06db-dc19-4a6b-a526-c5fa09933ed2,LIVIN_MANDIRI_REVIEWS,2,2025,kenapa di login tidak bisa ya malah muncul tul...
10,33535e95-15cb-49b3-bcb7-894957cb159d,WONDR_BNI_REVIEWS,1,2025,ngelag mulu deh
12,8091018d-801d-4a9f-a3ff-86491d781b1e,BRIMO_REVIEWS,1,2025,transaksi berhasil uang enggak masuk gimnaa si...
13,658c217f-74b2-4280-b07a-7ab529fd97a1,BCAMOBILE_REVIEWS,2,2025,sering keluar harus verifikasi lagi terus luma...
17,47ed6779-21fd-45bb-a5a6-c6d92ef93186,BCAMOBILE_REVIEWS,1,2025,malu ih bca mah


In [47]:
df["word_count"] = df["text"].astype(str).str.split().apply(len)
df = df[df["word_count"] >= 5].reset_index(drop=True)
print(f"Total documents setelah filter: {len(df):,}")

Total documents setelah filter: 42,661


In [48]:
documents = df["text"].astype(str).tolist()

print(f"Total documents : {len(documents):,}")

Total documents : 42,661


# SIMCSE IndoBERT

In [49]:
from sentence_transformers import SentenceTransformer

# Gunakan SimCSE untuk menekan anisotropy dan merapatkan klaster
embedding_model = SentenceTransformer("LazarusNLP/simcse-indobert-base", device=device)

embeddings = embedding_model.encode(
    documents,
    batch_size=128,             # GPU T4/V100 Colab sanggup menangani batch 128 untuk 60k data
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True   # Wajib: Memaksa vektor berukuran L2=1 agar Cosine Distance presisi
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/334 [00:00<?, ?it/s]

# BERTopic

In [100]:
# embeddings = np.load("indobert_embeddings.npy")

print("Embedding Shape :", embeddings.shape)

Embedding Shape : (42661, 768)


stop words

In [101]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [102]:
from nltk.corpus import stopwords as nltk_stopwords

sastrawi_stopwords = StopWordRemoverFactory().get_stop_words()

# Pure stopwords gabungan (NLTK + Sastrawi)
pure_stopwords = list(set(nltk_stopwords.words('indonesian')).union(set(sastrawi_stopwords)))

vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    stop_words=pure_stopwords,
    token_pattern=r"(?u)\b[^\d\W]+\b",
    min_df=5  # Untuk 60k data, min_df=5 efektif membuang kata typo langka
)

# Strict c-TF-IDF Transformer untuk memotong frequent words antar-klaster
ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True,
    bm25_weighting=True
)

baseline UMAP for testing purpose

In [103]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    metric="cosine",
    min_dist=0.0,
    random_state=42
)

baseline HDBSCAN

In [104]:
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=15,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

In [105]:
topic_model = BERTopic(
    embedding_model=None,
    calculate_probabilities=False,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
)

In [106]:
topics, probabilities = topic_model.fit_transform(
    documents,
    embeddings
)

2026-08-11 09:58:07,991 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-11 09:58:09,490 - BERTopic - Dimensionality - Completed ✓
2026-08-11 09:58:09,493 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-11 09:58:09,813 - BERTopic - Cluster - Completed ✓
2026-08-11 09:58:09,826 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-11 09:58:10,957 - BERTopic - Representation - Completed ✓


outliers removal

# Evaluation for Topic Quality

Basic Statistics

In [107]:
topic_info = topic_model.get_topic_info()

topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,21704,-1_aplikasi_nya_transaksi_bank,"[aplikasi, nya, transaksi, bank, masuk, banget...",[wkwk lucu nih perusahaan padahal sudah gede g...
1,0,2842,0_brimo_aplikasi brimo_akun brimo_buka,"[brimo, aplikasi brimo, akun brimo, buka, akun...","[kenapa brimo saya enggak bisa di buka, makin ..."
2,1,2564,1_wajah_verifikasi_verifikasi wajah_ktp,"[wajah, verifikasi, verifikasi wajah, ktp, gag...",[susah untuk selfie verifikasi wajah gagal ter...
3,2,1468,2_livin_aplikasi livin_mandiri_livin mandiri,"[livin, aplikasi livin, mandiri, livin mandiri...",[saya meminta tolong kenapa aplikasi livin man...
4,3,1386,3_potongan_biaya_admin_saldo,"[potongan, biaya, admin, saldo, ribu, uang, po...",[anjrit kenapa saldo saya dapat 2x potongan te...
5,4,863,4_wifi_jaringan_internet_sinyal,"[wifi, jaringan, internet, sinyal, bagus, paka...","[jaringan bagus tapi tetap tak bisa dibuka, to..."
6,5,600,5_qris_qris gagal_gagal saldo_saldo,"[qris, qris gagal, gagal saldo, saldo, gagal, ...","[transfer vua qris gagal tapi saldo terpotong,..."
7,6,569,6_bni_bni mobile_mobile_wonder,"[bni, bni mobile, mobile, wonder, banking, mob...","[masih lebih baik apk bni mobile, trial dahulu..."
8,7,567,7_pasword_password_salah_username,"[pasword, password, salah, username, sandi, lo...",[saya sudah isi pasword dan username dengan be...
9,8,538,8_merah_indikator_lampu_hijau,"[merah, indikator, lampu, hijau, indikator mer...","[sinyal bagus lampu indikator merah melulu, ke..."


In [108]:
num_topics = len(topic_info) - 1

outlier_count = (np.array(topics) == -1).sum()

outlier_percentage = (
    outlier_count / len(topics)
) * 100

print(f"Topics              : {num_topics}")
print(f"Outliers            : {outlier_count:,}")
print(f"Outlier Percentage  : {outlier_percentage:.2f}%")

Topics              : 63
Outliers            : 21,704
Outlier Percentage  : 50.88%


Topic Size

In [109]:
topic_info[["Topic","Count"]]

,Topic,Count
0,-1,21704
1,0,2842
2,1,2564
3,2,1468
4,3,1386
...,...,...
59,58,53
60,59,52
61,60,50
62,61,50


Top Words

In [110]:
top_10_topics = topic_model.get_topic_info()
top_10_topics = top_10_topics[top_10_topics.Topic != -1].nlargest(10, "Count")

for _, row in top_10_topics.iterrows():
    topic_id = row['Topic']
    doc_count = row['Count']

    print("=" * 80)
    print(f"TOPIC {topic_id} | JUMLAH DOKUMEN: {doc_count}")
    print("=" * 80)

    # Menampilkan word-score pair bawaan BERTopic (c-TF-IDF scores)
    words_with_scores = topic_model.get_topic(topic_id)
    for word, score in words_with_scores:
        print(f"  - {word:<20} : {score:.4f}")
    print()

TOPIC 0 | JUMLAH DOKUMEN: 2842
  - brimo                : 0.0929
  - aplikasi brimo       : 0.0351
  - akun brimo           : 0.0203
  - buka                 : 0.0160
  - akun                 : 0.0156
  - update               : 0.0155
  - aplikasi             : 0.0149
  - bri                  : 0.0145
  - apk brimo            : 0.0133
  - brimo nya            : 0.0133

TOPIC 1 | JUMLAH DOKUMEN: 2564
  - wajah                : 0.0809
  - verifikasi           : 0.0719
  - verifikasi wajah     : 0.0669
  - ktp                  : 0.0355
  - gagal                : 0.0315
  - foto                 : 0.0304
  - mata                 : 0.0300
  - wajah gagal          : 0.0282
  - susah                : 0.0220
  - foto ktp             : 0.0210

TOPIC 2 | JUMLAH DOKUMEN: 1468
  - livin                : 0.1118
  - aplikasi livin       : 0.0393
  - mandiri              : 0.0374
  - livin mandiri        : 0.0299
  - livin by             : 0.0273
  - by mandiri           : 0.0268
  - by               

Representative Reviews

In [111]:
# Ambil info topik dan urutkan berdasarkan jumlah dokumen terbesar (kecuali outlier -1)
topic_info = topic_model.get_topic_info()
top_10_topics = topic_info[topic_info.Topic != -1].nlargest(10, "Count")["Topic"].tolist()

print("=== TOP 10 TOPIK PALING REPRESENTATIF ===")

for topic_id in top_10_topics:
    # Ambil ukuran klaster asli
    cluster_size = topic_info.loc[topic_info.Topic == topic_id, "Count"].values[0]

    # Ambil kata kunci utama topik untuk mempermudah pembacaan aspek
    keywords = ", ".join([w for w, _ in topic_model.get_topic(topic_id)[:5]])

    # Ambil dokumen yang secara matematis paling dekat dengan centroid klaster (Bawaan BERTopic)
    rep_docs = topic_model.get_representative_docs(topic_id)

    print("\n" + "=" * 120)
    print(f"TOPIC {topic_id} | CLUSTER SIZE: {cluster_size}")
    print(f"KEYWORDS : {keywords}")
    print("=" * 120)

    # BERTopic menyimpan maksimum 3 representative docs per topik secara default
    for i, doc in enumerate(rep_docs, 1):
        print(f"{i}. {doc}")

=== TOP 10 TOPIK PALING REPRESENTATIF ===

TOPIC 0 | CLUSTER SIZE: 2842
KEYWORDS : brimo, aplikasi brimo, akun brimo, buka, akun
1. kenapa brimo saya enggak bisa di buka
2. makin di update enggak jelas brimo sering keluar
3. saya mau memperbaharui brimo mengapa tidak ada di plestor

TOPIC 1 | CLUSTER SIZE: 2564
KEYWORDS : wajah, verifikasi, verifikasi wajah, ktp, gagal
1. susah untuk selfie verifikasi wajah gagal terus
2. kenapa ketika verifikasi wajah enggak pernah bisa
3. saya mau tanya kenapa setelah verifikasi wajah kembali ke awal lagi

TOPIC 2 | CLUSTER SIZE: 1468
KEYWORDS : livin, aplikasi livin, mandiri, livin mandiri, livin by
1. saya meminta tolong kenapa aplikasi livin mandiri saya tidak bisa di gunakan
2. saya baru buat livin kenapa ada saldo terbelokir
3. livin tidak dapat di gunakan

TOPIC 3 | CLUSTER SIZE: 1386
KEYWORDS : potongan, biaya, admin, saldo, ribu
1. anjrit kenapa saldo saya dapat 2x potongan terus tiap bulan
2. admin bri ini terlalu banyak potongan nya ada ada

silhoutte score

In [112]:
from sklearn.metrics import silhouette_score

mask = np.array(topics) != -1

silhouette = silhouette_score(
    topic_model.umap_model.embedding_[mask],
    np.array(topics)[mask]
)

print(f"Silhouette Score : {silhouette:.4f}")

Silhouette Score : 0.5574


In [113]:
from itertools import chain

top_n = 10
topic_words = []

for topic in topic_info.Topic:
    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
    ]
    topic_words.append(words)

unique_words = len(
    set(chain.from_iterable(topic_words))
)

total_words = len(topic_words) * top_n
topic_diversity = unique_words / total_words

print(f"Topic Diversity : {topic_diversity:.4f}")

Topic Diversity : 0.7492


NPMI

In [114]:
analyzer = topic_model.vectorizer_model.build_analyzer()

In [115]:
doc.split()

['aplikasi',
 'nya',
 'sering',
 'force',
 'close',
 'sendiri',
 'sudah',
 'di',
 'update',
 'sudah',
 'di',
 'restart',
 'hp',
 'nya',
 'masih',
 'tetap',
 'sama',
 'sudah',
 'clear',
 'cache',
 'data',
 'dan',
 'download',
 'aplikasi',
 'ulang',
 'pun',
 'tetap',
 'sama']

In [116]:
tokenized_docs = [
    analyzer(doc)
    for doc in documents
]

In [117]:
from gensim.corpora import Dictionary

dictionary = Dictionary(tokenized_docs)
topic_words = []

for topic in topic_info.Topic:

    if topic == -1:
        continue

    words = []

    for word, score in topic_model.get_topic(topic):
        if word in dictionary.token2id:
            words.append(word)
    # Need at least 2 words for coherence
    if len(words) >= 2:
        topic_words.append(words)

In [118]:
# sanity check
print(f"Valid Topics : {len(topic_words)}")

print()

print(topic_words[:3])

Valid Topics : 63

[['brimo', 'aplikasi brimo', 'akun brimo', 'buka', 'akun', 'update', 'aplikasi', 'bri', 'apk brimo', 'brimo nya'], ['wajah', 'verifikasi', 'verifikasi wajah', 'ktp', 'gagal', 'foto', 'mata', 'wajah gagal', 'susah', 'foto ktp'], ['livin', 'aplikasi livin', 'mandiri', 'livin mandiri', 'livin by', 'by mandiri', 'by', 'login livin', 'pakai livin', 'masuk livin']]


In [119]:
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence="c_npmi"
)

npmi = coherence_model.get_coherence()
print(f"NPMI : {npmi:.4f}")

NPMI : 0.0979


# Evaluation for Clustering Quality


DBCV

In [120]:
import pandas as pd
from scipy.stats import chi2_contingency

df["topic"] = topics

# 1. Baseline: proporsi tiap bank di keseluruhan korpus
baseline = df["bank"].value_counts(normalize=True) * 100
print("Proporsi bank di keseluruhan korpus (baseline):")
print(baseline.round(2))
print()

# 2. Proporsi tiap bank DI DALAM tiap topik
crosstab = pd.crosstab(df["topic"], df["bank"], normalize="index") * 100
crosstab = crosstab.round(2)

# 3. Hitung "lift" = proporsi di topik / proporsi baseline
#    >1 artinya over-represented di topik itu, <1 artinya under-represented
lift = crosstab.copy()
for bank in baseline.index:
    lift[bank] = crosstab[bank] / baseline[bank]

# 4. Tandai topik yang "njomplang" (deviasi lift > 1.5x atau < 0.5x dari baseline)
def flag_imbalance(row):
    return any(row > 1.5) or any(row < 0.5)

lift["is_imbalanced"] = lift[baseline.index].apply(flag_imbalance, axis=1)

# gabung count per topik biar gampang liat mana yang topik "besar" (bukan cuma noise kecil)
topic_sizes = df[df["topic"] != -1]["topic"].value_counts()
lift["topic_size"] = lift.index.map(topic_sizes)

result = lift[lift.index != -1].sort_values("is_imbalanced", ascending=False)
print(result[list(baseline.index) + ["is_imbalanced", "topic_size"]])

Proporsi bank di keseluruhan korpus (baseline):
bank
BRIMO_REVIEWS            29.27
WONDR_BNI_REVIEWS        26.60
LIVIN_MANDIRI_REVIEWS    26.05
BCAMOBILE_REVIEWS        18.09
Name: proportion, dtype: float64

bank   BRIMO_REVIEWS  WONDR_BNI_REVIEWS  LIVIN_MANDIRI_REVIEWS  \
topic                                                            
0           3.335047           0.030456               0.025723   
29          0.801901           1.458129               1.116434   
1           0.744842           1.658538               0.853450   
32          2.180542           0.470000               0.580868   
34          1.372149           1.065961               1.027749   
...              ...                ...                    ...   
21          0.981278           1.300961               1.022758   
49          1.223864           1.346833               0.515602   
27          0.824110           1.058065               1.041954   
45          0.830943           1.016329               0.881860 